# Mixture of Experts

In this exercise, we will test the MoE model on a simple task. The task is admittedly slightly artificial - it is unfortunately not easy to build an NLP task which can be trained in < 1min on Colab and still see advantage of MoE. But let's take it as an exercise. :)

We have one binary classifier trained on a mix of two tasks. Both are about review sentiment, but on different
kinds of text, and with opposite labelling:

* Task "positive" is movie reviews and label 1 will mean "positive review"
* Task "complaint" is restaurant reviews and label 1 means "there is a complaint, i.e. negative review"

The model is never told which task an example belongs to; we will only use that afterwards to inspect the routing.



# There are two tasks:

1. Fill in the missing parts of the MoE model till it trains. For those who cannot make this work, I leave behind the outputs of my own run, so you can do the second task.
2. Answer, at the end of the notebook, the following questions:
   (a) Did MoE help in this simple case? Do you have an insight as to why the baseline model was so bad on this task?
   (b) The routing per task, per label and per sentiment almost always settles to the pattern shown in my outputs. In this pattern, what role does the router take? And what role does the expert take?
   (c) In terms of parameters, is this a fair comparison between the Baseline and the MoE as we have them?


# 1. Data prep etc...

In [2]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset, concatenate_datasets, Dataset
from transformers import Trainer, TrainingArguments
from sentence_transformers import SentenceTransformer

N_TRAIN = 1000   # per task, half label 1, half label 0
N_TEST = 400     # per task
TASKS = ["positive", "complaint"]

## Data
Each task is turned into a balanced binary dataset with columns `text`, `label` (0/1) and `task`.

In [3]:
def prepare(ds, task_id, n, positive_label):
    ds = ds.shuffle(seed=0)
    pos = ds.filter(lambda ex: ex["label"] == positive_label).select(range(n // 2))
    neg = ds.filter(lambda ex: ex["label"] != positive_label).select(range(n // 2))
    return Dataset.from_dict({"text": pos[:]["text"] + neg[:]["text"],
                              "label": [1] * (n // 2) + [0] * (n // 2),
                              "task": [task_id] * n})

imdb = load_dataset("stanfordnlp/imdb")          # orig label: 0 negative, 1 positive
yelp = load_dataset("fancyzhx/yelp_polarity")    # orig label: 0 negative, 1 positive
sources = [(imdb, 1),    # movies: label 1 = positive review
           (yelp, 0)]    # restaurants: label 1 = complaint = negative review

train_ds = concatenate_datasets([prepare(d["train"], i, N_TRAIN, p) for i, (d, p) in enumerate(sources)])
test_ds = concatenate_datasets([prepare(d["test"], i, N_TEST, p) for i, (d, p) in enumerate(sources)])
train_ds = train_ds.shuffle(seed=0)
test_ds = test_ds.shuffle(seed=0)
print(train_ds, test_ds)

README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/256M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/17.7M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/560000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/38000 [00:00<?, ? examples/s]

...

Filter:   0%|          | 0/25000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/25000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/38000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/38000 [00:00<?, ? examples/s]

Dataset({
    features: ['text', 'label', 'task'],
    num_rows: 2000
}) Dataset({
    features: ['text', 'label', 'task'],
    num_rows: 800
})


## Frozen embeddings
Sentence embeddings from a pretrained encoder, computed once. The encoder is not trained.

In [4]:
encoder = SentenceTransformer("intfloat/e5-base-v2")   # e5 models expect the prefix "query: "

def add_embeddings(ds):
    emb = encoder.encode(["query: " + t for t in ds[:]["text"]], batch_size=64,
                         normalize_embeddings=True, show_progress_bar=True)
    return ds.add_column("embeddings", emb.tolist())

train_ds = add_embeddings(train_ds)
test_ds = add_embeddings(test_ds)
train_ds.set_format("torch", columns=["embeddings", "label", "task"])
test_ds.set_format("torch", columns=["embeddings", "label", "task"])
DIM = train_ds[0]["embeddings"].shape[0] #we will need this later

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/650 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Flattening the indices:   0%|          | 0/2000 [00:00<?, ? examples/s]

Batches:   0%|          | 0/13 [00:00<?, ?it/s]

Flattening the indices:   0%|          | 0/800 [00:00<?, ? examples/s]

## Training and evaluation helpers

In [5]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    return {"accuracy": (logits.argmax(-1) == labels).mean()}

def train(model, epochs=30):
    args = TrainingArguments(output_dir="out", num_train_epochs=epochs, per_device_train_batch_size=64,
                             learning_rate=1e-3, eval_strategy="epoch", save_strategy="no",
                             report_to="none", logging_steps=50)
    trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=test_ds,
                      compute_metrics=compute_metrics)
    trainer.train()
    return trainer

def accuracy_per_task(trainer):
    pred = trainer.predict(test_ds).predictions.argmax(-1)
    labels, tasks = test_ds[:]["label"].numpy(), test_ds[:]["task"].numpy()
    for i, name in enumerate(TASKS):
        print(f"{name:10s} accuracy {(pred[tasks == i] == labels[tasks == i]).mean():.3f}")
    print(f"{'all':10s} accuracy {(pred == labels).mean():.3f}")

## Baseline: a linear classifier (on top of the frozen embeddings)

In [6]:
class LinearClassifier(nn.Module):
    def __init__(self, dim, n_classes=2):
        super().__init__()
        self.linear = nn.Linear(dim, n_classes)

    def forward(self, embeddings, labels=None):
        logits = self.linear(embeddings)
        #If given labels (means training), return also the loss
        loss = F.cross_entropy(logits, labels) if labels is not None else None
        return {"loss": loss, "logits": logits}

baseline = train(LinearClassifier(DIM))
accuracy_per_task(baseline)

/home/lauris/Documents/school/2026-27/Deep-Learning/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.682305,0.601250
2,0.685340,0.672827,0.617500
3,0.685340,0.664830,0.627500
4,0.670727,0.657624,0.640000
5,0.657679,0.651636,0.670000
6,0.657679,0.645170,0.660000
7,0.649235,0.639853,0.672500
8,0.638449,0.634771,0.680000
9,0.638449,0.630445,0.675000
10,0.630705,0.625903,0.698750


positive   accuracy 0.542
complaint  accuracy 0.960
all        accuracy 0.751


## Mixture of Experts

- **Experts**: `n_experts` linear classifiers, each like the baseline.
- **Router**: a linear layer giving one score per expert; softmax gives the routing probabilities.
- **Top-1 routing**: each example goes to its single most probable expert; the output is that expert's
  output scaled by its routing probability. (Scaling by the probability is what lets the router learn:
  it is the only path through which the task loss reaches the router.)
- **Load-balancing loss** (Switch Transformer): `n_experts * sum_i f_i * P_i`, where `f_i` is the fraction
  of examples routed to expert i and `P_i` the mean routing probability of expert i. Added to the loss
  with weight `aux_weight`.

NB! For simplicity of code every expert runs on every example here; the unselected experts just get weight 0.

In [ ]:
# This is a skeleton you can use, of course feel free to write your own!
# Make an effort to read & understand this code!

class MoEClassifier(nn.Module):
    def __init__(self, dim, n_experts=2, n_classes=2, aux_weight=0.1):
        super().__init__()
        self.n_experts, self.aux_weight = n_experts, aux_weight
        self.router = nn.Softmax(dim=dim)
        self.experts = nn.ModuleList([self.router for _ in range(n_experts)])

    def route(self, embeddings):
        probs = ... #TODO FILL HERE                        # (B, E)
        top_p, top_i = probs.max(dim=-1)                                           # (B,)
        gates = ... #TODO FILL HERE  # (B, E), zero except the chosen expert, p otherwise
        return gates, probs

    def forward(self, embeddings, labels=None):
        gates, probs = self.route(embeddings)
        expert_logits = torch.stack([e(embeddings) for e in self.experts], dim=1)   # (B, E, n_classes)
        logits = (gates.unsqueeze(-1) * expert_logits).sum(1)
        loss = None
        if labels is not None:
            f = F.one_hot(gates.argmax(-1), self.n_experts).float().mean(0)   # fraction routed to each expert
            balance = self.n_experts * (f * probs.mean(0)).sum()
            loss = ... + self.aux_weight * balance #TODO FILL HERE
        return {"loss": loss, "logits": logits}

moe = train(MoEClassifier(DIM, aux_weight=0.1))
accuracy_per_task(moe)

NameError: name 'FILL_HERE' is not defined

## Which expert gets which examples?
Rows = experts, cells = number of test examples routed to that expert. Columns: the task, the label, and the
underlying sentiment of the review (= the label, flipped back for the complaint task).

In [ ]:
@torch.no_grad()
def routing_table(trainer):
    model = trainer.model.eval()
    test = test_ds[:]
    expert = model.route(test["embeddings"].to(next(model.parameters()).device))[0].argmax(-1).cpu().numpy()
    tasks, labels = test["task"].numpy(), test["label"].numpy()
    print("expert x task", TASKS)
    for e in range(model.n_experts):
        print(f"expert {e}", np.bincount(tasks[expert == e], minlength=len(TASKS))) #bincount counts unique integers
    print("\nexpert x label [0, 1]")
    for e in range(model.n_experts):
        print(f"expert {e}", np.bincount(labels[expert == e], minlength=2))
    sentiment = np.where(tasks == 1, 1 - labels, labels)   # 1 = positive review
    print("\nexpert x sentiment [neg, pos]")
    for e in range(model.n_experts):
        print(f"expert {e}", np.bincount(sentiment[expert == e], minlength=2))

routing_table(moe)